## 18_b_w_estimation
  Destination exponent with GDP and active enterprises as destination mass, with the same
  bootstrap draws as 16; paired test of b_N against beta_W * b_W; simulation control.
  Output: data/processed/18_b_w_estimation/. Environment: nuts3-mobility.

In [ ]:
import os
import sys
from importlib.util import find_spec
from pathlib import Path

# PROJ data fix: must run BEFORE geopandas / pyproj / skmob are imported
def find_proj_dir():
    """Return the first directory that contains proj.db (no pyproj import needed)."""
    candidates = [Path(sys.prefix) / "share" / "proj"]  # conda environment
    spec = find_spec("pyproj")
    if spec is not None and spec.submodule_search_locations:
        pyproj_dir = Path(list(spec.submodule_search_locations)[0])
        candidates.append(pyproj_dir / "proj_dir" / "share" / "proj")
    for cand in candidates:
        if (cand / "proj.db").exists():
            return cand
    return None

proj_dir = find_proj_dir()
if proj_dir is not None:
    os.environ["PROJ_LIB"] = str(proj_dir)
    os.environ["PROJ_DATA"] = str(proj_dir)
    print("PROJ data: proj.db found, environment variables set")
else:
    print("PROJ data: proj.db NOT found in the expected places")

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely.wkb
from geopandas.array import GeometryArray

sys.path.append(str(Path.cwd().parent))  # notebook is in notebooks/
from src import config

# ----------------------------- CONFIG -----------------------------
YEAR = 2023                                   # same year as notebooks 04, 05, 05b, 16
COL_ID = "nuts3_id"
COL_COUNTRY, COL_YEAR = "country_code", "year"
CRS_GEO = "EPSG:4326"
NUTS3_TABLE = config.PROCESSED_DIR / "nuts3_master_table.parquet"

# Destination masses to fit; "population" reproduces the saved b of 05 / 05b (check)
MASS_COLS = ["population", "gdp_total", "active_enterprises"]
COL_POP = "population"

# Folders of the original fits and the file names saved there
SOURCE_DIRS = {
    "CZ": config.PROCESSED_DIR / "05_gravity_cz",
    "PL": config.PROCESSED_DIR / "05b_gravity_pl",
}
OD_FILE, PARAMS_FILE = "od_observed.csv", "gravity_params.csv"

# Fit sample rules per variant (same as notebook 16); the variant name is the tag of all output files
VARIANTS = {
    "CZ_nodiag": {"country": "CZ", "include_diagonal": False, "drop_zero_pairs": False, "main_fit": False},
    "PL_main":   {"country": "PL", "include_diagonal": False, "drop_zero_pairs": True,  "main_fit": True},
}

# Output folder of this notebook (checkpoints and result tables)
OUT_DIR = config.PROCESSED_DIR / "18_b_w_estimation"
# -------------------------------------------------------------------

COUNTRIES = list(dict.fromkeys(v["country"] for v in VARIANTS.values()))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Master table: geometry is stored as WKB bytes; build it element by element
nuts3 = pd.read_parquet(NUTS3_TABLE)
geom_arr = np.empty(len(nuts3), dtype=object)
for k, wkb_bytes in enumerate(nuts3["geometry"]):
    geom_arr[k] = shapely.wkb.loads(wkb_bytes) if wkb_bytes is not None else None
nuts3 = nuts3.drop(columns="geometry")
nuts3 = gpd.GeoDataFrame(nuts3, geometry=GeometryArray(geom_arr, crs=CRS_GEO))

regions, od_obs, main_params = {}, {}, {}
for c in COUNTRIES:
    # Regions of the country in the same order as in the original notebooks
    reg = nuts3[(nuts3[COL_COUNTRY] == c) & (nuts3[COL_YEAR] == YEAR)]
    reg = reg[[COL_ID, "geometry"] + MASS_COLS].reset_index(drop=True)

    # Observed OD matrix (rows = origin, columns = destination), aligned to the region order
    od = pd.read_csv(SOURCE_DIRS[c] / OD_FILE, index_col=0)
    assert set(od.index) == set(od.columns) == set(reg[COL_ID]), f"{c}: region ids differ"
    od = od.loc[reg[COL_ID], reg[COL_ID]]

    params = pd.read_csv(SOURCE_DIRS[c] / PARAMS_FILE)
    regions[c], od_obs[c], main_params[c] = reg, od, params

    # skmob drops zero-mass destinations, so missing or zero masses must be reported
    n_bad = {m: int((reg[m].isna() | (reg[m] <= 0)).sum()) for m in MASS_COLS}
    print(f"{c}: regions = {len(reg)} | missing or zero mass = {n_bad}"
          f" | commuters in matrix = {int(od.to_numpy().sum())}")
    print(f"{c}: main destination_exp =",
          params.set_index("model")["destination_exp"].round(4).to_dict())

# Size of the fit sample of each variant under its rules
for tag, rule in VARIANTS.items():
    od = od_obs[rule["country"]]
    mask = np.ones(od.shape, dtype=bool)
    if not rule["include_diagonal"]:
        np.fill_diagonal(mask, False)
    if rule["drop_zero_pairs"]:
        mask &= od.to_numpy() > 0
    params = main_params[rule["country"]]
    check = (f" | n_fit_pairs in {PARAMS_FILE} = {int(params['n_fit_pairs'].iloc[0])}"
             if rule["main_fit"] and "n_fit_pairs" in params.columns else "")
    print(f"{tag}: pairs in fit sample = {int(mask.sum())}{check}")

In [ ]:
import warnings

import skmob
from skmob.models.gravity import Gravity

# ----------------------------- CONFIG -----------------------------
TILE_ID = "tile_ID"
COL_FLOW_ORIGIN, COL_FLOW_DEST, COL_FLOW = "origin", "destination", "flow"
GRAVITY_TYPE = "singly constrained"
GRAVITY_VARIANTS = {"gravity_power": "power_law", "gravity_exp": "exponential"}
FULL_FIT_CSV = OUT_DIR / "full_fit.csv"
CHECK_TOLERANCE = 1e-6        # population b must match the saved b within this
# -------------------------------------------------------------------


def build_fdf(country, draw, rule):
    """Build the skmob fit sample for a draw of regions (positions, repeats allowed)."""
    reg = regions[country]
    od = od_obs[country].to_numpy()
    draw = np.asarray(draw)
    orig_ids = reg[COL_ID].to_numpy()[draw]
    slot_ids = np.array([f"{rid}#{k}" for k, rid in enumerate(orig_ids)])

    # Pair rules are applied on the ORIGINAL region ids (copies of one region count as diagonal)
    flows = od[np.ix_(draw, draw)]
    mask = np.ones(flows.shape, dtype=bool)
    if not rule["include_diagonal"]:
        mask &= orig_ids[:, None] != orig_ids[None, :]
    if rule["drop_zero_pairs"]:
        mask &= flows > 0
    o_pos, d_pos = np.nonzero(mask)

    flows_long = pd.DataFrame({
        COL_FLOW_ORIGIN: slot_ids[o_pos],
        COL_FLOW_DEST: slot_ids[d_pos],
        COL_FLOW: flows[o_pos, d_pos],
    })

    # Tessellation keeps the geometry and all mass columns (one row per slot)
    tess = reg.iloc[draw].reset_index(drop=True).rename(columns={COL_ID: TILE_ID})
    tess[TILE_ID] = slot_ids

    return skmob.FlowDataFrame(
        flows_long,
        origin=COL_FLOW_ORIGIN,
        destination=COL_FLOW_DEST,
        flow=COL_FLOW,
        tessellation=tess,
        tile_id=TILE_ID,
    )


def fit_b(fdf, mass_col):
    """Fit both gravity variants with the given destination mass; failures are recorded by class name."""
    out = {}
    for name, deterrence in GRAVITY_VARIANTS.items():
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            try:
                model = Gravity(gravity_type=GRAVITY_TYPE, deterrence_func_type=deterrence)
                model.fit(fdf, relevance_column=mass_col)
                b, err = model.destination_exp, ""
            except Exception as e:
                b, err = np.nan, type(e).__name__
        out[name] = {"b": b, "error_class": err,
                     "warnings": ";".join(sorted({w.category.__name__ for w in caught}))}
    return out


# Fit on the full sample (every region exactly once) for every mass column
rows = []
for tag, rule in VARIANTS.items():
    c = rule["country"]
    fdf_full = build_fdf(c, np.arange(len(regions[c])), rule)
    saved = main_params[c].set_index("model")["destination_exp"]   # b_N of notebooks 05 / 05b
    for mass_col in MASS_COLS:
        for model_name, r in fit_b(fdf_full, mass_col).items():
            rows.append({"variant": tag, "model": model_name, "mass_col": mass_col,
                         "n_pairs": len(fdf_full), "b": r["b"],
                         "b_saved": saved[model_name] if mass_col == COL_POP else np.nan,
                         "diff": r["b"] - saved[model_name] if mass_col == COL_POP else np.nan,
                         "error_class": r["error_class"], "warnings": r["warnings"]})

full_fit = pd.DataFrame(rows)
full_fit.to_csv(FULL_FIT_CSV, index=False)
print(full_fit.round(4).to_string(index=False))

# Check: the population fit must reproduce the saved b
pop = full_fit[full_fit["mass_col"] == COL_POP]
print("population b matches the saved b:", bool((pop["diff"].abs() < CHECK_TOLERANCE).all()))
print("Saved:", os.path.relpath(FULL_FIT_CSV))

In [ ]:
import time

# ----------------------------- CONFIG -----------------------------
# These three values must be the same as in notebook 16 (same draws, same seeds)
N_BOOT = 1000           # bootstrap draws per variant
CHUNK_SIZE = 20         # draws per checkpoint file
SEED_BASE = 0           # draw i uses seed SEED_BASE + i

BOOT_MASS_COLS = [m for m in MASS_COLS if m != COL_POP]   # b_N comes from notebook 16
BOOT_DIR = OUT_DIR / "boot_chunks"
BOOT_ALL_CSV = OUT_DIR / "boot_w_all.csv"
BOOT_SUMMARY_CSV = OUT_DIR / "boot_w_summary.csv"
# -------------------------------------------------------------------

BOOT_DIR.mkdir(parents=True, exist_ok=True)

for tag, rule in VARIANTS.items():
    c = rule["country"]
    n_reg = len(regions[c])
    for start in range(0, N_BOOT, CHUNK_SIZE):
        stop = min(start + CHUNK_SIZE, N_BOOT)
        chunk_csv = BOOT_DIR / f"bootw_{tag}_{start:04d}.csv"
        if chunk_csv.exists():
            continue                      # resume: this chunk is already done

        t0 = time.time()
        rows = []
        for i in range(start, stop):
            rng = np.random.default_rng(SEED_BASE + i)
            draw = rng.integers(0, n_reg, size=n_reg)   # regions drawn with replacement

            # A failed draw is recorded by class name, not raised
            try:
                fdf = build_fdf(c, draw, rule)
                n_pairs = len(fdf)
                res_by_mass = {m: fit_b(fdf, m) for m in BOOT_MASS_COLS}
            except Exception as e:
                n_pairs = np.nan
                res_by_mass = {m: {g: {"b": np.nan, "error_class": type(e).__name__, "warnings": ""}
                                   for g in GRAVITY_VARIANTS} for m in BOOT_MASS_COLS}

            for mass_col, res in res_by_mass.items():
                for model_name, r in res.items():
                    rows.append({"variant": tag, "draw": i, "model": model_name, "mass_col": mass_col,
                                 "n_unique_regions": len(set(draw.tolist())), "n_pairs": n_pairs,
                                 "b": r["b"], "error_class": r["error_class"],
                                 "warnings": r["warnings"]})

        pd.DataFrame(rows).to_csv(chunk_csv, index=False)
        print(f"{tag}: draws {start}-{stop - 1} done in {time.time() - t0:.1f} s")

n_files = sum(len(list(BOOT_DIR.glob(f"bootw_{tag}_*.csv"))) for tag in VARIANTS)
print("Chunk files:", n_files, "| expected:", len(VARIANTS) * -(-N_BOOT // CHUNK_SIZE))

# Combine all chunk files into one table
boot_w = pd.concat([pd.read_csv(p) for tag in VARIANTS
                    for p in sorted(BOOT_DIR.glob(f"bootw_{tag}_*.csv"))], ignore_index=True)
boot_w["error_class"] = boot_w["error_class"].fillna("")
boot_w.to_csv(BOOT_ALL_CSV, index=False)

# Summary per variant, model and mass: failed draws and the range of b
summary = (boot_w.groupby(["variant", "model", "mass_col"])
           .agg(n_draws=("draw", "count"),
                n_nan=("b", lambda s: int(s.isna().sum())),
                n_error=("error_class", lambda s: int((s != "").sum())),
                b_min=("b", "min"), b_median=("b", "median"), b_max=("b", "max"))
           .reset_index())
summary.to_csv(BOOT_SUMMARY_CSV, index=False)

print()
print(summary.round(4).to_string(index=False))
errors = (boot_w.loc[boot_w["error_class"] != "", ["variant", "model", "mass_col", "error_class"]]
          .value_counts().rename("count").reset_index())
print(errors.to_string(index=False) if len(errors) else "No error classes recorded")
print("Saved:", os.path.relpath(BOOT_ALL_CSV), "|", os.path.relpath(BOOT_SUMMARY_CSV))

In [ ]:
# ----------------------------- CONFIG -----------------------------
B_N_CSV = config.PROCESSED_DIR / "16_b_bootstrap" / "boot_all.csv"          # b with population mass
BETA_BOOT_CSV = config.PROCESSED_DIR / "16_b_bootstrap" / "boot_beta_all.csv"
BETA_SAMPLE = "full"                                                          # same sample as the b bootstrap here
# y variable of the scaling fit that belongs to each destination mass
BETA_Y_FOR_MASS = {m: m for m in BOOT_MASS_COLS}
H_CI_LEVEL = 0.95                                                             # interval of the difference
EQUIV_LEVEL = 0.90                                                            # 90% interval = TOST at 5%
DELTAS = [0.1, 0.2, 0.3, 0.5]                                                 # fixed margins, as in notebook 16
PAIRED_CSV = OUT_DIR / "paired_h_test.csv"
# -------------------------------------------------------------------

b_n = pd.read_csv(B_N_CSV).dropna(subset=["b"])[["variant", "draw", "model", "b"]].rename(columns={"b": "b_N"})
b_w = (pd.read_csv(BOOT_ALL_CSV).dropna(subset=["b"])
       [["variant", "draw", "model", "mass_col", "b"]].rename(columns={"b": "b_W"}))
beta = pd.read_csv(BETA_BOOT_CSV)
beta = beta[beta["sample"] == BETA_SAMPLE].dropna(subset=["beta"])[["variant", "draw", "y_variable", "beta"]]

q_lo, q_hi = (1 - H_CI_LEVEL) / 2, 1 - (1 - H_CI_LEVEL) / 2
e_lo, e_hi = (1 - EQUIV_LEVEL) / 2, 1 - (1 - EQUIV_LEVEL) / 2

rows = []
for mass_col, y_col in BETA_Y_FOR_MASS.items():
    # Draws where b_N, b_W and beta are all valid (inner joins on variant and draw)
    m = (b_w[b_w["mass_col"] == mass_col]
         .merge(b_n, on=["variant", "draw", "model"], how="inner")
         .merge(beta[beta["y_variable"] == y_col], on=["variant", "draw"], how="inner"))
    m["diff_H"] = m["b_N"] - m["beta"] * m["b_W"]        # H: b_N = beta_W * b_W
    m["diff_bW1"] = m["b_N"] - m["beta"]                 # special case b_W = 1 (notebook 16)

    for (tag, model_name), g in m.groupby(["variant", "model"]):
        lo, hi = g["diff_H"].quantile([q_lo, q_hi])
        el, eh = g["diff_H"].quantile([e_lo, e_hi])
        row = {"country": VARIANTS[tag]["country"], "model": model_name, "mass_col": mass_col,
               "n_pairs": len(g),
               "diff_H_median": g["diff_H"].median(), "diff_H_lo": lo, "diff_H_hi": hi,
               "zero_in_ci": bool(lo <= 0 <= hi),
               "diff_bW1_median": g["diff_bW1"].median(),
               "abs_median_H": abs(g["diff_H"].median()),
               "abs_median_bW1": abs(g["diff_bW1"].median()),
               "diff_lo90": el, "diff_hi90": eh, "delta_min": max(abs(el), abs(eh))}
        for d in DELTAS:
            row[f"equiv_delta_{d}"] = bool(-d <= el and eh <= d)
        rows.append(row)

paired_h = (pd.DataFrame(rows)
            .sort_values(["country", "model", "mass_col"]).reset_index(drop=True))
paired_h.to_csv(PAIRED_CSV, index=False)

show = ["country", "model", "mass_col", "n_pairs", "diff_H_median", "diff_H_lo", "diff_H_hi",
        "zero_in_ci", "diff_bW1_median", "delta_min"] + [f"equiv_delta_{d}" for d in DELTAS]
print(paired_h[show].round(3).to_string(index=False))
print("Saved:", os.path.relpath(PAIRED_CSV))

In [ ]:
# ----------------------------- CONFIG -----------------------------
FIT_MASS_COLS = [m for m in MASS_COLS if m != COL_POP]      # masses regressed on population
SAVED_BETA_CSV = Path("../results/scaling_by_country/beta_by_country_2023.csv")
SAVED_SAMPLE = "full"
CHECK_TOLERANCE = 1e-6
FIT_CSV = OUT_DIR / "mass_vs_population_fit.csv"
# -------------------------------------------------------------------

saved = pd.read_csv(SAVED_BETA_CSV)

rows = []
for c in COUNTRIES:
    reg = regions[c]
    x = np.log(reg[COL_POP].to_numpy(dtype=float))
    for mass_col in FIT_MASS_COLS:
        y = np.log(reg[mass_col].to_numpy(dtype=float))

        # OLS of ln W on ln N: slope, intercept, residuals
        slope, intercept = np.polyfit(x, y, 1)
        resid = y - (intercept + slope * x)
        ss_res = float((resid ** 2).sum())
        ss_tot = float(((y - y.mean()) ** 2).sum())

        ref = saved[(saved["group"] == c) & (saved["sample"] == SAVED_SAMPLE)
                    & (saved["y_variable"] == mass_col)]["beta"]
        beta_saved = float(ref.iloc[0]) if len(ref) else np.nan

        rows.append({"country": c, "mass_col": mass_col, "n_regions": len(reg),
                     "beta": slope, "r_squared": 1 - ss_res / ss_tot,
                     "resid_std": resid.std(ddof=2),            # two fitted parameters
                     "beta_saved": beta_saved, "diff_to_saved": slope - beta_saved})

fit_table = pd.DataFrame(rows)
fit_table.to_csv(FIT_CSV, index=False)
print(fit_table.round(4).to_string(index=False))
print("beta matches the saved beta of notebook 14:",
      bool((fit_table["diff_to_saved"].abs() < CHECK_TOLERANCE).all()))
print("Saved:", os.path.relpath(FIT_CSV))

In [ ]:
# ----------------------------- CONFIG -----------------------------
CTRL_MASS_COLS = [m for m in MASS_COLS if m != COL_POP]   # masses to replace by an exact power of N
CTRL_SUFFIX = "_synth"
N_CTRL_DRAWS = 100                                         # first draws of the bootstrap (same seeds as notebook 16)
SEED_BASE = 0                                              # must match notebooks 16 and 18
B_N_CSV = config.PROCESSED_DIR / "16_b_bootstrap" / "boot_all.csv"
CTRL_CSV = OUT_DIR / "control_synthetic_mass.csv"
# -------------------------------------------------------------------

if CTRL_CSV.exists():
    print("skip, already exists:", os.path.relpath(CTRL_CSV))
    ctrl = pd.read_csv(CTRL_CSV)
else:
    # 1) Synthetic masses: exact power law of the population, residual u = 0
    slopes = {}
    for c in COUNTRIES:
        reg = regions[c]
        x = np.log(reg[COL_POP].to_numpy(dtype=float))
        for m in CTRL_MASS_COLS:
            y = np.log(reg[m].to_numpy(dtype=float))
            slope, intercept = np.polyfit(x, y, 1)
            reg[m + CTRL_SUFFIX] = np.exp(intercept + slope * x)
            slopes[(c, m)] = slope

    # 2) b_N references: full-sample b from the full fit, draws from notebook 16
    full_ref = pd.read_csv(FULL_FIT_CSV)
    full_ref = full_ref[full_ref["mass_col"] == COL_POP].set_index(["variant", "model"])["b"]
    b_n_draws = (pd.read_csv(B_N_CSV).dropna(subset=["b"])
                 .set_index(["variant", "draw", "model"])["b"])

    rows = []
    for tag, rule in VARIANTS.items():
        c = rule["country"]
        n_reg = len(regions[c])

        # Full sample
        fdf_full = build_fdf(c, np.arange(n_reg), rule)
        for m in CTRL_MASS_COLS:
            for model_name, r in fit_b(fdf_full, m + CTRL_SUFFIX).items():
                rows.append({"variant": tag, "draw": -1, "model": model_name, "mass_col": m,
                             "b_N": full_ref[(tag, model_name)], "b_W_synth": r["b"],
                             "beta_fit": slopes[(c, m)], "error_class": r["error_class"]})

        # First N_CTRL_DRAWS bootstrap draws
        for i in range(N_CTRL_DRAWS):
            rng = np.random.default_rng(SEED_BASE + i)
            draw = rng.integers(0, n_reg, size=n_reg)
            try:
                fdf = build_fdf(c, draw, rule)
                res_by_mass = {m: fit_b(fdf, m + CTRL_SUFFIX) for m in CTRL_MASS_COLS}
            except Exception as e:
                res_by_mass = {m: {g: {"b": np.nan, "error_class": type(e).__name__}
                                   for g in GRAVITY_VARIANTS} for m in CTRL_MASS_COLS}
            for m, res in res_by_mass.items():
                for model_name, r in res.items():
                    key = (tag, i, model_name)
                    rows.append({"variant": tag, "draw": i, "model": model_name, "mass_col": m,
                                 "b_N": b_n_draws.get(key, np.nan), "b_W_synth": r["b"],
                                 "beta_fit": slopes[(c, m)], "error_class": r["error_class"]})

    ctrl = pd.DataFrame(rows)
    ctrl["diff_ctrl"] = ctrl["b_N"] - ctrl["beta_fit"] * ctrl["b_W_synth"]
    ctrl.to_csv(CTRL_CSV, index=False)
    print("Saved:", os.path.relpath(CTRL_CSV))

# Summary: how far from zero is b_N - beta * b_W when u = 0 exactly?
summ = (ctrl.dropna(subset=["diff_ctrl"])
        .groupby(["variant", "model", "mass_col"])["diff_ctrl"]
        .agg(n="count", max_abs=lambda s: s.abs().max(), median="median")
        .reset_index())
print(summ.round(6).to_string(index=False))
full_rows = ctrl[ctrl["draw"] == -1][["variant", "model", "mass_col", "b_N", "b_W_synth", "beta_fit", "diff_ctrl"]]
print()
print("Full sample:")
print(full_rows.round(6).to_string(index=False))

In [ ]:
# ----------------------------- CONFIG -----------------------------
OUTLIER_THRESHOLD = 0.001          # |b_N - beta * b_W_synth| above this counts as an outlier
OUTLIER_COUNTRY = "PL"             # country whose draws are checked
OUTLIER_CSV = OUT_DIR / "control_outliers.csv"
# -------------------------------------------------------------------

ctrl = pd.read_csv(CTRL_CSV)
boot_info = pd.read_csv(BOOT_ALL_CSV)[["variant", "draw", "model", "mass_col",
                                       "n_unique_regions", "n_pairs", "warnings"]]

# Keep the bootstrap draws of the chosen country (the full-sample rows have draw = -1)
sel = ctrl[(ctrl["draw"] >= 0)
           & (ctrl["variant"].map(lambda t: VARIANTS[t]["country"]) == OUTLIER_COUNTRY)].copy()
sel = sel.merge(boot_info, on=["variant", "draw", "model", "mass_col"], how="left")
sel["is_outlier"] = sel["diff_ctrl"].abs() > OUTLIER_THRESHOLD
sel["has_singular_warning"] = sel["warnings"].fillna("").str.contains("SingularMatrixWarning")

# Count outliers among the draws with a valid difference
valid = sel.dropna(subset=["diff_ctrl"])
count = (valid.groupby(["variant", "model", "mass_col"])
         .agg(n_valid=("diff_ctrl", "count"), n_outlier=("is_outlier", "sum"))
         .reset_index())
print(count.to_string(index=False))

# Compare outliers with the other draws: unique regions, pairs, warnings
cmp_cols = (valid.groupby(["is_outlier"])
            .agg(n=("diff_ctrl", "count"),
                 unique_regions_median=("n_unique_regions", "median"),
                 unique_regions_min=("n_unique_regions", "min"),
                 pairs_median=("n_pairs", "median"),
                 share_singular_warning=("has_singular_warning", "mean"))
            .reset_index())
print()
print(cmp_cols.round(3).to_string(index=False))

valid[valid["is_outlier"]].to_csv(OUTLIER_CSV, index=False)
print("Saved:", os.path.relpath(OUTLIER_CSV))

In [ ]:
# ----------------------------- CONFIG -----------------------------
FIRST_DRAWS = N_CTRL_DRAWS                      # the control exists only for these draws
KEY = ["variant", "draw", "model", "mass_col"]  # one outlier = one (variant, draw, model, mass) case
RERUN_CSV = OUT_DIR / "paired_h_test_no_outliers.csv"
# B_N_CSV, BETA_BOOT_CSV, BETA_SAMPLE, BETA_Y_FOR_MASS, H_CI_LEVEL, EQUIV_LEVEL, DELTAS,
# BOOT_ALL_CSV and OUTLIER_CSV come from the earlier cells
# -------------------------------------------------------------------

outliers = pd.read_csv(OUTLIER_CSV)[KEY].assign(is_outlier=True)

b_n = (pd.read_csv(B_N_CSV).dropna(subset=["b"])
       [["variant", "draw", "model", "b"]].rename(columns={"b": "b_N"}))
b_w = (pd.read_csv(BOOT_ALL_CSV).dropna(subset=["b"])
       [["variant", "draw", "model", "mass_col", "b"]].rename(columns={"b": "b_W"}))
beta = pd.read_csv(BETA_BOOT_CSV)
beta = beta[beta["sample"] == BETA_SAMPLE].dropna(subset=["beta"])[["variant", "draw", "y_variable", "beta"]]

q_lo, q_hi = (1 - H_CI_LEVEL) / 2, 1 - (1 - H_CI_LEVEL) / 2
e_lo, e_hi = (1 - EQUIV_LEVEL) / 2, 1 - (1 - EQUIV_LEVEL) / 2

rows = []
for mass_col, y_col in BETA_Y_FOR_MASS.items():
    m = (b_w[b_w["mass_col"] == mass_col]
         .merge(b_n, on=["variant", "draw", "model"], how="inner")
         .merge(beta[beta["y_variable"] == y_col], on=["variant", "draw"], how="inner"))
    m = m[m["draw"] < FIRST_DRAWS].copy()
    m["diff_H"] = m["b_N"] - m["beta"] * m["b_W"]

    # Mark the cases that are outliers in the control
    m = m.merge(outliers, on=KEY, how="left")
    m["is_outlier"] = m["is_outlier"].fillna(False).astype(bool)

    versions = {"all_first_draws": m, "no_outliers": m[~m["is_outlier"]]}
    for version, g_all in versions.items():
        for (tag, model_name), g in g_all.groupby(["variant", "model"]):
            lo, hi = g["diff_H"].quantile([q_lo, q_hi])
            el, eh = g["diff_H"].quantile([e_lo, e_hi])
            row = {"country": VARIANTS[tag]["country"], "model": model_name, "mass_col": mass_col,
                   "version": version, "n_pairs": len(g),
                   "diff_H_median": g["diff_H"].median(), "diff_H_lo": lo, "diff_H_hi": hi,
                   "zero_in_ci": bool(lo <= 0 <= hi), "delta_min": max(abs(el), abs(eh))}
            for d in DELTAS:
                row[f"equiv_delta_{d}"] = bool(-d <= el and eh <= d)
            rows.append(row)

rerun = (pd.DataFrame(rows)
         .sort_values(["country", "model", "mass_col", "version"]).reset_index(drop=True))
rerun.to_csv(RERUN_CSV, index=False)

show = ["country", "model", "mass_col", "version", "n_pairs", "diff_H_median",
        "diff_H_lo", "diff_H_hi", "zero_in_ci", "delta_min"]
print(rerun[show].round(3).to_string(index=False))
print("Saved:", os.path.relpath(RERUN_CSV))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ----------------------------- CONFIG -----------------------------
FIG_DIR = Path("../figures") / "b_w_estimation"
FIG_FILE = "h_difference_hist.png"
FIG_DPI = 150
FIG_WIDTH, ROW_HEIGHT = 11, 2.6                 # figure width and height of one row
N_HIST_BINS = 40
X_CLIP_PCT = (0.5, 99.5)                        # x range per country from these percentiles of the differences
X_PAD = 0.05                                    # padding of the x range (fraction of the range)
MODEL_LABELS = {"gravity_power": "Power-law deterrence", "gravity_exp": "Exponential deterrence"}
HIST_COLOR, CI_COLOR = "steelblue", "lightgray"
BW1_COLOR = "tab:red"
# B_N_CSV, BOOT_ALL_CSV, BETA_BOOT_CSV, BETA_SAMPLE, BETA_Y_FOR_MASS and H_CI_LEVEL come from the earlier cells
# -------------------------------------------------------------------

FIG_DIR.mkdir(parents=True, exist_ok=True)

b_n = (pd.read_csv(B_N_CSV).dropna(subset=["b"])
       [["variant", "draw", "model", "b"]].rename(columns={"b": "b_N"}))
b_w = (pd.read_csv(BOOT_ALL_CSV).dropna(subset=["b"])
       [["variant", "draw", "model", "mass_col", "b"]].rename(columns={"b": "b_W"}))
beta = pd.read_csv(BETA_BOOT_CSV)
beta = beta[beta["sample"] == BETA_SAMPLE].dropna(subset=["beta"])[["variant", "draw", "y_variable", "beta"]]

q_lo, q_hi = (1 - H_CI_LEVEL) / 2, 1 - (1 - H_CI_LEVEL) / 2

# Per-draw differences for every mass column (draws where b_N, b_W and beta are all valid)
parts = []
for mass_col, y_col in BETA_Y_FOR_MASS.items():
    m = (b_w[b_w["mass_col"] == mass_col]
         .merge(b_n, on=["variant", "draw", "model"], how="inner")
         .merge(beta[beta["y_variable"] == y_col], on=["variant", "draw"], how="inner"))
    m["diff_H"] = m["b_N"] - m["beta"] * m["b_W"]        # H: b_N = beta_W * b_W
    m["diff_bW1"] = m["b_N"] - m["beta"]                 # special case b_W = 1 (notebook 16)
    parts.append(m)
diffs = pd.concat(parts, ignore_index=True)

tags = list(VARIANTS)
masses = list(BETA_Y_FOR_MASS)
models = list(GRAVITY_VARIANTS)
row_keys = [(tag, mass) for tag in tags for mass in masses]

fig, axes = plt.subplots(len(row_keys), len(models),
                         figsize=(FIG_WIDTH, ROW_HEIGHT * len(row_keys)), squeeze=False)

# Common x range inside a country (both models and masses), so the panels are comparable
x_range = {}
for tag in tags:
    d = diffs.loc[diffs["variant"] == tag, ["diff_H", "diff_bW1"]].to_numpy().ravel()
    lo, hi = np.percentile(d, X_CLIP_PCT)
    pad = X_PAD * (hi - lo)
    x_range[tag] = (lo - pad, hi + pad)

for r, (tag, mass_col) in enumerate(row_keys):
    for j, model_name in enumerate(models):
        ax = axes[r][j]
        g = diffs[(diffs["variant"] == tag) & (diffs["mass_col"] == mass_col)
                  & (diffs["model"] == model_name)]
        lo, hi = g["diff_H"].quantile([q_lo, q_hi])

        ax.axvspan(lo, hi, color=CI_COLOR, alpha=0.6)
        ax.hist(g["diff_H"], bins=N_HIST_BINS, range=x_range[tag], color=HIST_COLOR, alpha=0.85)
        ax.axvline(0, color="black", lw=2)
        ax.axvline(g["diff_H"].median(), color="black", lw=1.5, ls="--")
        ax.axvline(g["diff_bW1"].median(), color=BW1_COLOR, lw=1.5, ls=":")
        ax.set_xlim(*x_range[tag])
        ax.text(0.02, 0.95, f"n = {len(g)}", transform=ax.transAxes, va="top")

        if r == 0:
            ax.set_title(MODEL_LABELS[model_name])
        if j == 0:
            ax.set_ylabel(f"{VARIANTS[tag]['country']}, {mass_col}\nnumber of draws")
        if r == len(row_keys) - 1:
            ax.set_xlabel("b_N - beta_W * b_W")

handles = [
    Patch(facecolor=CI_COLOR, alpha=0.6, label=f"{int(H_CI_LEVEL * 100)}% percentile interval"),
    Line2D([0], [0], color="black", lw=2, label="zero (H holds exactly)"),
    Line2D([0], [0], color="black", lw=1.5, ls="--", label="median of b_N - beta_W * b_W"),
    Line2D([0], [0], color=BW1_COLOR, lw=1.5, ls=":", label="median of b_N - beta_W (b_W = 1, notebook 16)"),
]
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False)

fig.savefig(FIG_DIR / FIG_FILE, dpi=FIG_DPI, bbox_inches="tight")
plt.show()
print("Saved:", os.path.relpath(FIG_DIR / FIG_FILE))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ----------------------------- CONFIG -----------------------------
FIG_DIR = Path("../figures") / "b_w_estimation"
FIG_FILE = "b_n_vs_beta_bw_hist.png"
FIG_DPI = 150
FIG_WIDTH, BLOCK_HEIGHT = 11, 4.6              # figure width and height of one country block
HEIGHT_RATIOS = (3, 2)                         # histogram row : prediction row
BLOCK_GAP, ROW_GAP, COL_GAP = 0.45, 0.12, 0.12 # spacing between blocks, rows inside a block, columns
BOTTOM_MARGIN = 0.12                           # room for the legend under the last x label
N_HIST_BINS = 40
X_PAD = 0.05                                   # padding of the x range (fraction of the range)
MODEL_LABELS = {"gravity_power": "Power-law deterrence", "gravity_exp": "Exponential deterrence"}
X_LABEL, Y_LABEL = "exponent (b_N or beta_W * b_W)", "number of draws"
CI_COLOR, HIST_COLOR = "lightgray", "steelblue"
PRED_COLOR, NO_OVERLAP_COLOR = "black", "tab:red"
# B_N_CSV, BOOT_ALL_CSV, BETA_BOOT_CSV, BETA_SAMPLE, BETA_Y_FOR_MASS, H_CI_LEVEL, FULL_FIT_CSV
# and COL_POP come from the earlier cells
# -------------------------------------------------------------------

FIG_DIR.mkdir(parents=True, exist_ok=True)
q_lo, q_hi = (1 - H_CI_LEVEL) / 2, 1 - (1 - H_CI_LEVEL) / 2

# b_N: all valid draws of notebook 16 and the full-sample fit of this notebook
b_n_all = pd.read_csv(B_N_CSV).dropna(subset=["b"])[["variant", "draw", "model", "b"]]
b_n_main = (pd.read_csv(FULL_FIT_CSV).query("mass_col == @COL_POP")
            .set_index(["variant", "model"])["b"])

# Predicted value beta_W * b_W per draw (draws where b_N is not needed, only b_W and beta)
b_w = (pd.read_csv(BOOT_ALL_CSV).dropna(subset=["b"])
       [["variant", "draw", "model", "mass_col", "b"]].rename(columns={"b": "b_W"}))
beta = pd.read_csv(BETA_BOOT_CSV)
beta = beta[beta["sample"] == BETA_SAMPLE].dropna(subset=["beta"])[["variant", "draw", "y_variable", "beta"]]

rows = []
for mass_col, y_col in BETA_Y_FOR_MASS.items():
    m = (b_w[b_w["mass_col"] == mass_col]
         .merge(beta[beta["y_variable"] == y_col], on=["variant", "draw"], how="inner"))
    m["pred"] = m["beta"] * m["b_W"]
    for (tag, model_name), g in m.groupby(["variant", "model"]):
        lo, hi = g["pred"].quantile([q_lo, q_hi])
        b_sel = b_n_all[(b_n_all["variant"] == tag) & (b_n_all["model"] == model_name)]["b"]
        b_lo, b_hi = b_sel.quantile([q_lo, q_hi])
        rows.append({"variant": tag, "model": model_name, "mass_col": mass_col,
                     "pred_median": g["pred"].median(), "pred_lo": lo, "pred_hi": hi,
                     "b_lo": b_lo, "b_hi": b_hi,
                     "ci_overlap": bool((lo <= b_hi) and (b_lo <= hi))})
pred = pd.DataFrame(rows)

tags = list(VARIANTS)
models = list(GRAVITY_VARIANTS)
masses = list(BETA_Y_FOR_MASS)
y_pos = {v: i for i, v in enumerate(masses)}

fig = plt.figure(figsize=(FIG_WIDTH, BLOCK_HEIGHT * len(tags)))
outer = fig.add_gridspec(len(tags), 1, hspace=BLOCK_GAP)

for k, tag in enumerate(tags):
    country = VARIANTS[tag]["country"]
    pred_c = pred[pred["variant"] == tag]

    # Two rows (histogram, prediction) and one column per model inside the country block
    inner = outer[k].subgridspec(2, len(models), height_ratios=list(HEIGHT_RATIOS),
                                 hspace=ROW_GAP, wspace=COL_GAP)

    # Common x range inside a country block (b_N draws and predicted intervals)
    b_all = b_n_all.loc[b_n_all["variant"] == tag, "b"]
    x_lo = min(b_all.min(), pred_c["pred_lo"].min())
    x_hi = max(b_all.max(), pred_c["pred_hi"].max())
    pad = X_PAD * (x_hi - x_lo)

    for j, model_name in enumerate(models):
        ax_h = fig.add_subplot(inner[0, j])
        ax_f = fig.add_subplot(inner[1, j], sharex=ax_h)

        b_valid = b_n_all[(b_n_all["variant"] == tag) & (b_n_all["model"] == model_name)]["b"]
        rows_pred = pred_c[pred_c["model"] == model_name]
        b_lo, b_hi = rows_pred.iloc[0][["b_lo", "b_hi"]]
        b_main = b_n_main[(tag, model_name)]

        # b_N interval and full-sample b_N run through both rows
        for ax in (ax_h, ax_f):
            ax.axvspan(b_lo, b_hi, color=CI_COLOR, alpha=0.6)
            ax.axvline(b_main, color="black", lw=2)

        # Top row: bootstrap distribution of b_N
        ax_h.hist(b_valid, bins=N_HIST_BINS, color=HIST_COLOR, alpha=0.85)
        ax_h.axvline(b_valid.median(), color="black", lw=1.5, ls="--")
        ax_h.text(0.02, 0.95, f"valid: {len(b_valid)}", transform=ax_h.transAxes, va="top")
        ax_h.set_title(f"{country} - {MODEL_LABELS[model_name]}")
        plt.setp(ax_h.get_xticklabels(), visible=False)
        if j == 0:
            ax_h.set_ylabel(Y_LABEL)

        # Bottom row: beta_W * b_W with its percentile interval (red = no overlap with the b_N interval)
        for _, r in rows_pred.iterrows():
            color = PRED_COLOR if r["ci_overlap"] else NO_OVERLAP_COLOR
            ax_f.errorbar(r["pred_median"], y_pos[r["mass_col"]],
                          xerr=[[r["pred_median"] - r["pred_lo"]], [r["pred_hi"] - r["pred_median"]]],
                          fmt="o", color=color, capsize=3)
        ax_f.set_yticks(range(len(masses)))
        ax_f.set_yticklabels(masses if j == 0 else [])
        ax_f.set_ylim(len(masses) - 0.5, -0.5)
        ax_f.set_xlabel(X_LABEL)

        ax_h.set_xlim(x_lo - pad, x_hi + pad)        # shared with the prediction row

# One shared legend under the last row
handles = [
    Patch(facecolor=CI_COLOR, alpha=0.6, label="b_N: 95% percentile interval"),
    Line2D([0], [0], color="black", lw=2, label="b_N (full-sample fit)"),
    Line2D([0], [0], color="black", lw=1.5, ls="--", label="b_N: bootstrap median"),
    Line2D([0], [0], marker="o", color=PRED_COLOR, label="beta_W * b_W with 95% percentile interval"),
    Line2D([0], [0], marker="o", color=NO_OVERLAP_COLOR,
           label="prediction interval does not overlap the b_N interval"),
]
fig.subplots_adjust(bottom=BOTTOM_MARGIN, top=0.95, left=0.2, right=0.97)
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False)

fig.savefig(FIG_DIR / FIG_FILE, dpi=FIG_DPI, bbox_inches="tight")
plt.show()
print("Saved:", os.path.relpath(FIG_DIR / FIG_FILE))

In [ ]:
import time

# ----------------------------- CONFIG -----------------------------
CAPITAL_IDS = {"CZ": "CZ010", "PL": "PL911"}       # as in notebooks 05, 05b and 16
B_N_REFIT_CSV = config.PROCESSED_DIR / "16_b_bootstrap" / "capital_excluded_refit.csv"
NOCAP_FULL_CSV = OUT_DIR / "full_fit_nocapital.csv"
NOCAP_DIR = OUT_DIR / "boot_chunks_nocapital"
NOCAP_ALL_CSV = OUT_DIR / "boot_w_nocapital_all.csv"
CHECK_TOLERANCE = 1e-6
# N_BOOT, CHUNK_SIZE, SEED_BASE, BOOT_MASS_COLS, MASS_COLS and COL_POP come from the earlier cells
# -------------------------------------------------------------------

NOCAP_DIR.mkdir(parents=True, exist_ok=True)

# 1) Full-sample fit without the capital, all masses; check population b against notebook 16
ref = pd.read_csv(B_N_REFIT_CSV).set_index(["variant", "model"])["b_without_capital"]
rows = []
for tag, rule in VARIANTS.items():
    c = rule["country"]
    ids = regions[c][COL_ID].to_numpy()
    non_cap_pos = np.flatnonzero(ids != CAPITAL_IDS[c])      # every non-capital region once
    fdf_nc = build_fdf(c, non_cap_pos, rule)
    for mass_col in MASS_COLS:
        for model_name, r in fit_b(fdf_nc, mass_col).items():
            rows.append({"variant": tag, "model": model_name, "mass_col": mass_col,
                         "n_regions": len(non_cap_pos), "n_pairs": len(fdf_nc), "b": r["b"],
                         "b_ref_16": ref[(tag, model_name)] if mass_col == COL_POP else np.nan,
                         "error_class": r["error_class"]})
nocap_full = pd.DataFrame(rows)
nocap_full["diff"] = nocap_full["b"] - nocap_full["b_ref_16"]
nocap_full.to_csv(NOCAP_FULL_CSV, index=False)
print(nocap_full.round(4).to_string(index=False))
pop = nocap_full[nocap_full["mass_col"] == COL_POP]
print("population b matches notebook 16:", bool((pop["diff"].abs() < CHECK_TOLERANCE).all()))

# 2) Bootstrap without the capital (same seeds and draw rule as notebook 16)
for tag, rule in VARIANTS.items():
    c = rule["country"]
    ids = regions[c][COL_ID].to_numpy()
    non_cap_pos = np.flatnonzero(ids != CAPITAL_IDS[c])
    n_nc = len(non_cap_pos)

    for start in range(0, N_BOOT, CHUNK_SIZE):
        stop = min(start + CHUNK_SIZE, N_BOOT)
        chunk_csv = NOCAP_DIR / f"bootwnocap_{tag}_{start:04d}.csv"
        if chunk_csv.exists():
            continue                                         # resume: this chunk is already done

        t0 = time.time()
        rows = []
        for i in range(start, stop):
            rng = np.random.default_rng(SEED_BASE + i)
            draw = non_cap_pos[rng.integers(0, n_nc, size=n_nc)]   # non-capital regions, with replacement

            # A failed draw is recorded by class name, not raised
            try:
                fdf = build_fdf(c, draw, rule)
                n_pairs = len(fdf)
                res_by_mass = {m: fit_b(fdf, m) for m in BOOT_MASS_COLS}
            except Exception as e:
                n_pairs = np.nan
                res_by_mass = {m: {g: {"b": np.nan, "error_class": type(e).__name__, "warnings": ""}
                                   for g in GRAVITY_VARIANTS} for m in BOOT_MASS_COLS}

            for mass_col, res in res_by_mass.items():
                for model_name, r in res.items():
                    rows.append({"variant": tag, "draw": i, "model": model_name, "mass_col": mass_col,
                                 "n_unique_regions": len(set(draw.tolist())), "n_pairs": n_pairs,
                                 "b": r["b"], "error_class": r["error_class"],
                                 "warnings": r["warnings"]})

        pd.DataFrame(rows).to_csv(chunk_csv, index=False)
        print(f"{tag} (no capital): draws {start}-{stop - 1} done in {time.time() - t0:.1f} s")

boot_w_nc = pd.concat([pd.read_csv(p) for tag in VARIANTS
                       for p in sorted(NOCAP_DIR.glob(f"bootwnocap_{tag}_*.csv"))], ignore_index=True)
boot_w_nc["error_class"] = boot_w_nc["error_class"].fillna("")
boot_w_nc.to_csv(NOCAP_ALL_CSV, index=False)

summary_nc = (boot_w_nc.groupby(["variant", "model", "mass_col"])
              .agg(n_draws=("draw", "count"), n_error=("error_class", lambda s: int((s != "").sum())),
                   b_min=("b", "min"), b_median=("b", "median"), b_max=("b", "max"))
              .reset_index())
print()
print(summary_nc.round(4).to_string(index=False))
print("Saved:", os.path.relpath(NOCAP_FULL_CSV), "|", os.path.relpath(NOCAP_ALL_CSV))

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
B_N_NOCAP_CSV = config.PROCESSED_DIR / "16_b_bootstrap" / "boot_nocapital_all.csv"
SAMPLES = {   # sample key (as in notebook 16) -> b_N draws, b_W draws, row title
    "full":        {"b_n": B_N_CSV,        "b_w": BOOT_ALL_CSV,   "label": "full sample (capital included)"},
    "no_capitals": {"b_n": B_N_NOCAP_CSV,  "b_w": NOCAP_ALL_CSV,  "label": "no capital"},
}
BY_SAMPLE_CSV = OUT_DIR / "paired_h_by_sample.csv"
FIG_PAIRED_FILE = "paired_h_difference.png"
MODEL_COLORS = {"gravity_power": "#0072B2", "gravity_exp": "#D55E00"}
MODEL_LABELS = {"gravity_power": "b, power law", "gravity_exp": "b, exponential"}
DODGE = 0.14          # vertical offset between the two models in a row
PANEL_WIDTH, PANEL_HEIGHT = 5.4, 3.6
# H_CI_LEVEL, EQUIV_LEVEL, DELTAS, BETA_BOOT_CSV, BETA_Y_FOR_MASS, FIG_DIR come from the earlier cells
# -------------------------------------------------------------------

q_lo, q_hi = (1 - H_CI_LEVEL) / 2, 1 - (1 - H_CI_LEVEL) / 2
e_lo, e_hi = (1 - EQUIV_LEVEL) / 2, 1 - (1 - EQUIV_LEVEL) / 2

beta_all = pd.read_csv(BETA_BOOT_CSV).dropna(subset=["beta"])

rows = []
for sample, src in SAMPLES.items():
    b_n = (pd.read_csv(src["b_n"]).dropna(subset=["b"])
           [["variant", "draw", "model", "b"]].rename(columns={"b": "b_N"}))
    b_w = (pd.read_csv(src["b_w"]).dropna(subset=["b"])
           [["variant", "draw", "model", "mass_col", "b"]].rename(columns={"b": "b_W"}))
    beta = beta_all[beta_all["sample"] == sample][["variant", "draw", "y_variable", "beta"]]

    for mass_col, y_col in BETA_Y_FOR_MASS.items():
        # Draws where b_N, b_W and beta are all valid
        m = (b_w[b_w["mass_col"] == mass_col]
             .merge(b_n, on=["variant", "draw", "model"], how="inner")
             .merge(beta[beta["y_variable"] == y_col], on=["variant", "draw"], how="inner"))
        m["diff_H"] = m["b_N"] - m["beta"] * m["b_W"]

        for (tag, model_name), g in m.groupby(["variant", "model"]):
            lo, hi = g["diff_H"].quantile([q_lo, q_hi])
            el, eh = g["diff_H"].quantile([e_lo, e_hi])
            row = {"country": VARIANTS[tag]["country"], "sample": sample, "model": model_name,
                   "mass_col": mass_col, "n_pairs": len(g),
                   "diff_H_median": g["diff_H"].median(), "diff_H_lo": lo, "diff_H_hi": hi,
                   "zero_in_ci": bool(lo <= 0 <= hi), "delta_min": max(abs(el), abs(eh))}
            for d in DELTAS:
                row[f"equiv_delta_{d}"] = bool(-d <= el and eh <= d)
            rows.append(row)

by_sample = (pd.DataFrame(rows)
             .sort_values(["country", "sample", "model", "mass_col"]).reset_index(drop=True))
by_sample.to_csv(BY_SAMPLE_CSV, index=False)
print(by_sample[["country", "sample", "model", "mass_col", "n_pairs", "diff_H_median",
                 "diff_H_lo", "diff_H_hi", "zero_in_ci", "delta_min"]].round(3).to_string(index=False))
print("Saved:", os.path.relpath(BY_SAMPLE_CSV))

# Figure: rows = samples, columns = countries (same layout as notebook 16)
countries_p = list(dict.fromkeys(by_sample["country"]))
fig, axes = plt.subplots(len(SAMPLES), len(countries_p),
                         figsize=(PANEL_WIDTH * len(countries_p), PANEL_HEIGHT * len(SAMPLES)),
                         sharex="col", squeeze=False)
offsets = {m: o for m, o in zip(MODEL_COLORS, (-DODGE, DODGE))}

for r, (sample, src) in enumerate(SAMPLES.items()):
    for k, c in enumerate(countries_p):
        ax = axes[r][k]
        sub = by_sample[(by_sample["country"] == c) & (by_sample["sample"] == sample)]
        masses = sorted(sub["mass_col"].unique())
        y_base = np.arange(len(masses))[::-1]                        # first mass on top

        for model_name, color in MODEL_COLORS.items():
            s = sub[sub["model"] == model_name].set_index("mass_col").loc[masses]
            ax.errorbar(s["diff_H_median"], y_base + offsets[model_name],
                        xerr=[s["diff_H_median"] - s["diff_H_lo"], s["diff_H_hi"] - s["diff_H_median"]],
                        fmt="o", color=color, capsize=2.5, markersize=5, lw=1.4,
                        label=MODEL_LABELS[model_name])

        ax.axvline(0, color="black", ls="--", lw=1)
        ax.set_yticks(y_base)
        ax.set_yticklabels(masses)
        ax.set_title(f"{c}: {src['label']}", fontsize=10)
        ax.grid(axis="x", alpha=0.3)
        if r == len(SAMPLES) - 1:
            ax.set_xlabel("b_N - beta_W * b_W (median and 95% interval)")

# One shared legend below the panels
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1))

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_path = FIG_DIR / FIG_PAIRED_FILE
fig.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", os.path.relpath(fig_path))

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
CO_MASS = "gdp_total"                 # change to "active_enterprises" to redraw
MODEL_COLORS = {"gravity_power": "#0072B2", "gravity_exp": "#D55E00"}
MODEL_LABELS = {"gravity_power": "b_N, power law", "gravity_exp": "b_N, exponential"}
POINT_ALPHA = 0.25
PANEL_WIDTH, PANEL_HEIGHT = 3.6, 3.4
SPEARMAN_CSV = OUT_DIR / "spearman_comodement.csv"
FIG_CO_FILE = f"b_n_vs_beta_bw_comodement_{CO_MASS}.png"
# SAMPLES, B_N_CSV, B_N_NOCAP_CSV, BOOT_ALL_CSV, NOCAP_ALL_CSV, BETA_BOOT_CSV, BETA_Y_FOR_MASS,
# VARIANTS and FIG_DIR come from the earlier cells
# -------------------------------------------------------------------

beta_all = pd.read_csv(BETA_BOOT_CSV).dropna(subset=["beta"])

# Per-draw table for every (variant, sample): b_N, beta_W * b_W (draws where all three are valid)
panels = {}
spearman_rows = []
for sample, src in SAMPLES.items():
    b_n = (pd.read_csv(src["b_n"]).dropna(subset=["b"])
           [["variant", "draw", "model", "b"]].rename(columns={"b": "b_N"}))
    b_w = (pd.read_csv(src["b_w"]).dropna(subset=["b"])
           [["variant", "draw", "model", "mass_col", "b"]].rename(columns={"b": "b_W"}))
    beta = beta_all[(beta_all["sample"] == sample)
                    & (beta_all["y_variable"] == BETA_Y_FOR_MASS[CO_MASS])][["variant", "draw", "beta"]]
    m = (b_w[b_w["mass_col"] == CO_MASS]
         .merge(b_n, on=["variant", "draw", "model"], how="inner")
         .merge(beta, on=["variant", "draw"], how="inner"))
    m["pred"] = m["beta"] * m["b_W"]
    for tag in VARIANTS:
        g_tag = m[m["variant"] == tag]
        panels[(VARIANTS[tag]["country"], sample)] = g_tag
        for model_name in MODEL_COLORS:
            g = g_tag[g_tag["model"] == model_name]
            spearman_rows.append({"country": VARIANTS[tag]["country"], "sample": sample,
                                  "model": model_name, "mass_col": CO_MASS, "n_pairs": len(g),
                                  "spearman_bN_pred": g["b_N"].corr(g["pred"], method="spearman")})

spearman = pd.DataFrame(spearman_rows)
spearman.to_csv(SPEARMAN_CSV, index=False)
print(spearman.round(3).to_string(index=False))
print("Saved:", os.path.relpath(SPEARMAN_CSV))

# Figure: rows = models, columns = (country, sample), as in notebook 16
col_keys = [(rule["country"], s) for rule in VARIANTS.values() for s in SAMPLES]
fig, axes = plt.subplots(len(MODEL_COLORS), len(col_keys),
                         figsize=(PANEL_WIDTH * len(col_keys), PANEL_HEIGHT * len(MODEL_COLORS)),
                         squeeze=False)

for r, model_name in enumerate(MODEL_COLORS):
    for k, (c, sample) in enumerate(col_keys):
        ax = axes[r][k]
        g = panels[(c, sample)]
        g = g[g["model"] == model_name]
        ax.scatter(g["pred"], g["b_N"], s=8, alpha=POINT_ALPHA, color=MODEL_COLORS[model_name], lw=0)

        # b_N = beta_W * b_W reference line over the shared range of the panel
        lo = min(g["pred"].min(), g["b_N"].min()); hi = max(g["pred"].max(), g["b_N"].max())
        ax.plot([lo, hi], [lo, hi], color="black", ls="--", lw=0.8)

        rho = g["b_N"].corr(g["pred"], method="spearman")
        ax.text(0.04, 0.94, f"Spearman = {rho:.2f}\nn = {len(g)}", transform=ax.transAxes,
                va="top", fontsize=9, bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
        if r == 0:
            ax.set_title(f"{c}: {SAMPLES[sample]['label']}", fontsize=10)
        if r == len(MODEL_COLORS) - 1:
            ax.set_xlabel(f"beta_W * b_W ({CO_MASS})")
        if k == 0:
            ax.set_ylabel(MODEL_LABELS[model_name])
        ax.grid(alpha=0.25)

fig.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_path = FIG_DIR / FIG_CO_FILE
fig.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", os.path.relpath(fig_path))

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
MODEL_COLORS = {"gravity_power": "#0072B2", "gravity_exp": "#D55E00"}
MODEL_LABELS = {"gravity_power": "b_N, power law", "gravity_exp": "b_N, exponential"}
DODGE = 0.14
PANEL_WIDTH, PANEL_HEIGHT = 5.0, 3.4
OVERVIEW_CSV = OUT_DIR / "spearman_overview.csv"
FIG_OVERVIEW_FILE = "spearman_overview.png"
# SAMPLES, BETA_BOOT_CSV, BETA_Y_FOR_MASS, VARIANTS and FIG_DIR come from the earlier cells
# -------------------------------------------------------------------

beta_all = pd.read_csv(BETA_BOOT_CSV).dropna(subset=["beta"])

rows = []
for sample, src in SAMPLES.items():
    b_n = (pd.read_csv(src["b_n"]).dropna(subset=["b"])
           [["variant", "draw", "model", "b"]].rename(columns={"b": "b_N"}))
    b_w = (pd.read_csv(src["b_w"]).dropna(subset=["b"])
           [["variant", "draw", "model", "mass_col", "b"]].rename(columns={"b": "b_W"}))
    for mass_col, y_col in BETA_Y_FOR_MASS.items():
        beta = beta_all[(beta_all["sample"] == sample)
                        & (beta_all["y_variable"] == y_col)][["variant", "draw", "beta"]]
        # Draws where b_N, b_W and beta are all valid
        m = (b_w[b_w["mass_col"] == mass_col]
             .merge(b_n, on=["variant", "draw", "model"], how="inner")
             .merge(beta, on=["variant", "draw"], how="inner"))
        m["pred"] = m["beta"] * m["b_W"]
        for (tag, model_name), g in m.groupby(["variant", "model"]):
            rows.append({"country": VARIANTS[tag]["country"], "sample": sample, "model": model_name,
                         "mass_col": mass_col, "n_pairs": len(g),
                         "spearman_bN_pred": g["b_N"].corr(g["pred"], method="spearman")})

overview = (pd.DataFrame(rows)
            .sort_values(["country", "sample", "model", "mass_col"]).reset_index(drop=True))
overview.to_csv(OVERVIEW_CSV, index=False)
print(overview.round(3).to_string(index=False))
print("Saved:", os.path.relpath(OVERVIEW_CSV))

# Figure: rows = samples, columns = countries (same layout as notebook 16)
countries_s = list(dict.fromkeys(overview["country"]))
fig, axes = plt.subplots(len(SAMPLES), len(countries_s),
                         figsize=(PANEL_WIDTH * len(countries_s), PANEL_HEIGHT * len(SAMPLES)),
                         sharex=True, squeeze=False)
offsets = {m: o for m, o in zip(MODEL_COLORS, (-DODGE, DODGE))}

for r, (sample, src) in enumerate(SAMPLES.items()):
    for k, c in enumerate(countries_s):
        ax = axes[r][k]
        sub = overview[(overview["country"] == c) & (overview["sample"] == sample)]
        masses = sorted(sub["mass_col"].unique())
        y_base = np.arange(len(masses))[::-1]                        # first mass on top

        for model_name, color in MODEL_COLORS.items():
            s = sub[sub["model"] == model_name].set_index("mass_col").loc[masses]
            ax.scatter(s["spearman_bN_pred"], y_base + offsets[model_name], color=color, s=40,
                       label=MODEL_LABELS[model_name], zorder=3)

        ax.axvline(0, color="black", ls="--", lw=1)
        ax.set_yticks(y_base)
        ax.set_yticklabels(masses)
        ax.set_title(f"{c}: {src['label']}", fontsize=10)
        ax.grid(axis="x", alpha=0.3)
        if r == len(SAMPLES) - 1:
            ax.set_xlabel("Spearman correlation of b_N and beta_W * b_W")

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1))

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_path = FIG_DIR / FIG_OVERVIEW_FILE
fig.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", os.path.relpath(fig_path))

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
CAPITAL_COUNT_MAX = 2                   # draws with this many or more capital copies are one group
MIN_DRAWS = 30                          # minimum draws in a stratum to report a correlation
STRATUM_STYLE = {0: ("#009E73", "o", "0 capital copies"),      # key = capital_copies value
                 1: ("#CC79A7", "s", "1 capital copy"),
                 2: ("#000000", "^", "2+ capital copies")}
MODEL_ROWS = {"gravity_power": "b_N, power law", "gravity_exp": "b_N, exponential"}
DODGE = 0.17
PANEL_WIDTH, PANEL_HEIGHT = 5.0, 3.4
STRATA_CSV = OUT_DIR / "spearman_by_capital_count.csv"
FIG_STRATA_FILE = "spearman_by_capital_count.png"
# B_N_CSV, BOOT_ALL_CSV, BETA_BOOT_CSV, BETA_Y_FOR_MASS, VARIANTS, CAPITAL_IDS, N_BOOT, SEED_BASE
# and FIG_DIR come from the earlier cells
# -------------------------------------------------------------------

# Number of capital copies per draw (full-sample draw rule, same seeds as the bootstrap)
count_rows = []
for tag, rule in VARIANTS.items():
    c = rule["country"]
    ids = regions[c][COL_ID].to_numpy()
    for i in range(N_BOOT):
        rng = np.random.default_rng(SEED_BASE + i)
        draw = rng.integers(0, len(ids), size=len(ids))
        n_cap = int(np.sum(ids[draw] == CAPITAL_IDS[c]))
        count_rows.append({"variant": tag, "draw": i, "capital_copies": min(n_cap, CAPITAL_COUNT_MAX)})
counts = pd.DataFrame(count_rows)

b_n = (pd.read_csv(B_N_CSV).dropna(subset=["b"])
       [["variant", "draw", "model", "b"]].rename(columns={"b": "b_N"}))
b_w = (pd.read_csv(BOOT_ALL_CSV).dropna(subset=["b"])
       [["variant", "draw", "model", "mass_col", "b"]].rename(columns={"b": "b_W"}))
beta_all = pd.read_csv(BETA_BOOT_CSV).dropna(subset=["beta"])
beta_full = beta_all[beta_all["sample"] == "full"][["variant", "draw", "y_variable", "beta"]]

rows = []
for mass_col, y_col in BETA_Y_FOR_MASS.items():
    # Draws where b_N, b_W and beta are all valid
    m = (b_w[b_w["mass_col"] == mass_col]
         .merge(b_n, on=["variant", "draw", "model"], how="inner")
         .merge(beta_full[beta_full["y_variable"] == y_col], on=["variant", "draw"], how="inner")
         .merge(counts, on=["variant", "draw"], how="inner"))
    m["pred"] = m["beta"] * m["b_W"]
    for (tag, model_name, k), g in m.groupby(["variant", "model", "capital_copies"]):
        rho = g["b_N"].corr(g["pred"], method="spearman") if len(g) >= MIN_DRAWS else np.nan
        rows.append({"country": VARIANTS[tag]["country"], "model": model_name, "mass_col": mass_col,
                     "capital_copies": k, "n_draws": len(g), "spearman_bN_pred": rho})
strata = pd.DataFrame(rows)
strata.to_csv(STRATA_CSV, index=False)

# Compact view: one row per country, model, mass; one column per stratum
idx = ["country", "model", "mass_col"]
view = strata.pivot_table(index=idx, columns="capital_copies", values="spearman_bN_pred")
view.columns = [f"rho_copies_{k}" for k in view.columns]
n_view = strata.pivot_table(index=idx, columns="capital_copies", values="n_draws")
n_view.columns = [f"n_copies_{k}" for k in n_view.columns]
print(view.join(n_view).round(2).to_string())
print("Saved:", os.path.relpath(STRATA_CSV))

# Figure: rows = models, columns = countries (same layout as notebook 16)
countries_k = list(dict.fromkeys(strata["country"]))
fig, axes = plt.subplots(len(MODEL_ROWS), len(countries_k),
                         figsize=(PANEL_WIDTH * len(countries_k), PANEL_HEIGHT * len(MODEL_ROWS)),
                         sharex=True, squeeze=False)
offsets = {k: o for k, o in zip(STRATUM_STYLE, (-DODGE, 0, DODGE))}

for r, (model_name, model_label) in enumerate(MODEL_ROWS.items()):
    for j, c in enumerate(countries_k):
        ax = axes[r][j]
        sub = strata[(strata["country"] == c) & (strata["model"] == model_name)]
        masses = sorted(sub["mass_col"].unique())
        y_base = np.arange(len(masses))[::-1]                        # first mass on top

        for k, (color, marker, label) in STRATUM_STYLE.items():
            s = sub[sub["capital_copies"] == k].set_index("mass_col").reindex(masses)
            ax.scatter(s["spearman_bN_pred"], y_base + offsets[k], color=color, marker=marker,
                       s=36, label=label, zorder=3)

        ax.axvline(0, color="black", ls="--", lw=1)
        ax.set_yticks(y_base)
        ax.set_yticklabels(masses)
        ax.set_title(f"{c}: {model_label}", fontsize=10)
        ax.grid(axis="x", alpha=0.3)
        if r == len(MODEL_ROWS) - 1:
            ax.set_xlabel("Spearman correlation of b_N and beta_W * b_W")

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1))

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_path = FIG_DIR / FIG_STRATA_FILE
fig.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", os.path.relpath(fig_path))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ----------------------------- CONFIG -----------------------------
B_N_NOCAP_CI_CSV = config.PROCESSED_DIR / "16_b_bootstrap" / "b_nocapital_ci.csv"
NOCAP_SAMPLE = "no_capitals"
MODEL_COLORS = {"gravity_power": "#0072B2", "gravity_exp": "#D55E00"}
MODEL_LABELS = {"gravity_power": "power law", "gravity_exp": "exponential"}
BAND_ALPHA = 0.12
DODGE = 0.14
NO_OVERLAP_EDGE = "red"
NEUTRAL_COLOR = "#555555"                  # color of the generic legend entries
PANEL_WIDTH, PANEL_HEIGHT = 5.6, 4.0
LEGEND_COLUMNS = 2
CMP_CSV = OUT_DIR / "b_n_vs_pred_no_capital.csv"
FIG_CMP_FILE = "b_n_vs_pred_no_capital.png"
# NOCAP_ALL_CSV, BETA_BOOT_CSV, BETA_Y_FOR_MASS, VARIANTS, H_CI_LEVEL and FIG_DIR come from the earlier cells
# -------------------------------------------------------------------

q_lo, q_hi = (1 - H_CI_LEVEL) / 2, 1 - (1 - H_CI_LEVEL) / 2

b_n_ci = pd.read_csv(B_N_NOCAP_CI_CSV)                       # b_N without the capital (notebook 16)
b_w = (pd.read_csv(NOCAP_ALL_CSV).dropna(subset=["b"])
       [["variant", "draw", "model", "mass_col", "b"]].rename(columns={"b": "b_W"}))
beta = pd.read_csv(BETA_BOOT_CSV)
beta = beta[beta["sample"] == NOCAP_SAMPLE].dropna(subset=["beta"])[["variant", "draw", "y_variable", "beta"]]

# beta_W * b_W per draw (draws where b_W and beta are valid), then median and interval
rows = []
for mass_col, y_col in BETA_Y_FOR_MASS.items():
    m = (b_w[b_w["mass_col"] == mass_col]
         .merge(beta[beta["y_variable"] == y_col], on=["variant", "draw"], how="inner"))
    m["pred"] = m["beta"] * m["b_W"]
    for (tag, model_name), g in m.groupby(["variant", "model"]):
        lo, hi = g["pred"].quantile([q_lo, q_hi])
        ref = b_n_ci[(b_n_ci["variant"] == tag) & (b_n_ci["model"] == model_name)].iloc[0]
        rows.append({"country": VARIANTS[tag]["country"], "model": model_name, "mass_col": mass_col,
                     "n_draws": len(g), "pred_median": g["pred"].median(), "pred_lo": lo, "pred_hi": hi,
                     "b_N_nocap": ref["b_without_capital"], "b_lo": ref["b_lo"], "b_hi": ref["b_hi"],
                     "ci_overlap": bool((lo <= ref["b_hi"]) and (ref["b_lo"] <= hi))})
cmp_nc = (pd.DataFrame(rows)
          .sort_values(["country", "model", "mass_col"]).reset_index(drop=True))
cmp_nc.to_csv(CMP_CSV, index=False)
print(cmp_nc.round(3).to_string(index=False))
print("Saved:", os.path.relpath(CMP_CSV))

# Figure: one panel per country (same style as notebook 16)
countries_c = list(dict.fromkeys(cmp_nc["country"]))
fig, axes = plt.subplots(1, len(countries_c),
                         figsize=(PANEL_WIDTH * len(countries_c), PANEL_HEIGHT), squeeze=False)
offsets = {m: o for m, o in zip(MODEL_COLORS, (-DODGE, DODGE))}

for ax, c in zip(axes[0], countries_c):
    sub = cmp_nc[cmp_nc["country"] == c]
    masses = sorted(sub["mass_col"].unique())
    y_base = np.arange(len(masses))[::-1]                            # first mass on top

    for model_name, color in MODEL_COLORS.items():
        s = sub[sub["model"] == model_name].set_index("mass_col").loc[masses]
        # b_N band and full-sample line (same for every mass)
        ax.axvspan(s["b_lo"].iloc[0], s["b_hi"].iloc[0], color=color, alpha=BAND_ALPHA, lw=0)
        ax.axvline(s["b_N_nocap"].iloc[0], color=color, lw=1.8)
        # beta_W * b_W per mass, red edge if no overlap with the b_N interval
        ax.errorbar(s["pred_median"], y_base + offsets[model_name],
                    xerr=[s["pred_median"] - s["pred_lo"], s["pred_hi"] - s["pred_median"]],
                    fmt="none", ecolor=color, capsize=3, lw=1.4)
        ax.scatter(s["pred_median"], y_base + offsets[model_name], color=color, s=40, zorder=3,
                   edgecolors=[NO_OVERLAP_EDGE if not o else color for o in s["ci_overlap"]],
                   linewidths=2)

    ax.set_yticks(y_base)
    ax.set_yticklabels(masses)
    ax.set_xlabel("exponent (b_N or beta_W * b_W)")
    ax.set_title(f"{c}: no capital", fontsize=10)
    ax.grid(axis="x", alpha=0.3)

# One shared legend below the panels: colors = models, shapes = what is drawn
handles = [Line2D([0], [0], color=col, lw=1.8, label=f"b_N, {MODEL_LABELS[m]} (full-sample fit)")
           for m, col in MODEL_COLORS.items()]
handles += [
    Patch(facecolor=NEUTRAL_COLOR, alpha=BAND_ALPHA * 2, label=f"b_N: {int(H_CI_LEVEL * 100)}% percentile interval (band)"),
    Line2D([0], [0], marker="o", color=NEUTRAL_COLOR, lw=1.4, markersize=6,
           label=f"beta_W * b_W: median and {int(H_CI_LEVEL * 100)}% interval"),
    Line2D([0], [0], marker="o", color="white", markeredgecolor=NO_OVERLAP_EDGE, markeredgewidth=2,
           markersize=7, lw=0, label="red edge: interval does not overlap the b_N band"),
]
fig.legend(handles=handles, loc="lower center", ncol=LEGEND_COLUMNS, frameon=False)
fig.tight_layout(rect=(0, 0.16, 1, 1))              # room for the two-row legend

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_path = FIG_DIR / FIG_CMP_FILE
fig.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", os.path.relpath(fig_path))

In [ ]:
import time

# ----------------------------- CONFIG -----------------------------
SIM_MASS = "gdp_total"                  # mass to replace by a power of population plus noise
N_SIM = 20                              # number of noise realizations
N_SIM_DRAWS = 100                       # first bootstrap draws used (same seeds and draws as before)
SIM_SEED_BASE = 1000                    # noise realization s uses seed SIM_SEED_BASE + s
SIM_COL = "w_sim"                       # temporary mass column in the region tables
SIM_DIR = OUT_DIR / "simulation_chunks"
SIM_SUMMARY_CSV = OUT_DIR / "simulation_summary.csv"
OBSERVED_CSV = OUT_DIR / "paired_h_test_no_outliers.csv"
OBSERVED_VERSION = "all_first_draws"
# fit_table (cell 5), B_N_CSV, SEED_BASE, GRAVITY_VARIANTS, VARIANTS, build_fdf and fit_b come from earlier cells
# -------------------------------------------------------------------

SIM_DIR.mkdir(parents=True, exist_ok=True)

# b_N draws of notebook 16 (population mass, full sample)
b_n_draws = (pd.read_csv(B_N_CSV).dropna(subset=["b"])
             .set_index(["variant", "draw", "model"])["b"])

for tag, rule in VARIANTS.items():
    c = rule["country"]
    reg = regions[c]
    n_reg = len(reg)
    x = np.log(reg[COL_POP].to_numpy(dtype=float))
    y = np.log(reg[SIM_MASS].to_numpy(dtype=float))
    slope, intercept = np.polyfit(x, y, 1)                       # same line as in cell 5
    sigma = float(fit_table.loc[(fit_table["country"] == c)
                                & (fit_table["mass_col"] == SIM_MASS), "resid_std"].iloc[0])

    for s in range(N_SIM):
        chunk_csv = SIM_DIR / f"sim_{tag}_{s:03d}.csv"
        if chunk_csv.exists():
            continue                                             # resume: this realization is done

        t0 = time.time()
        u = np.random.default_rng(SIM_SEED_BASE + s).normal(0.0, sigma, size=n_reg)
        ln_w = intercept + slope * x + u
        reg[SIM_COL] = np.exp(ln_w)

        rows = []
        for i in range(N_SIM_DRAWS):
            draw = np.random.default_rng(SEED_BASE + i).integers(0, n_reg, size=n_reg)
            beta_sim = np.polyfit(x[draw], ln_w[draw], 1)[0]     # scaling slope on the drawn regions
            try:
                res = fit_b(build_fdf(c, draw, rule), SIM_COL)
            except Exception as e:
                res = {m: {"b": np.nan, "error_class": type(e).__name__} for m in GRAVITY_VARIANTS}
            for model_name, r in res.items():
                b_n = b_n_draws.get((tag, i, model_name), np.nan)
                rows.append({"variant": tag, "sim": s, "draw": i, "model": model_name,
                             "b_N": b_n, "b_W_sim": r["b"], "beta_sim": beta_sim,
                             "diff_sim": b_n - beta_sim * r["b"], "error_class": r["error_class"]})
        pd.DataFrame(rows).to_csv(chunk_csv, index=False)
        print(f"{tag}: noise realization {s} done in {time.time() - t0:.1f} s")

    reg.drop(columns=SIM_COL, inplace=True, errors="ignore")      # keep the region table clean

# Summary: median difference per realization, then its spread across realizations
sim = pd.concat([pd.read_csv(p) for p in sorted(SIM_DIR.glob("sim_*.csv"))], ignore_index=True)
per_sim = (sim.dropna(subset=["diff_sim"])
           .groupby(["variant", "model", "sim"])["diff_sim"].median().rename("median_diff").reset_index())
spread = (per_sim.groupby(["variant", "model"])["median_diff"]
          .agg(sim_median="median", sim_p05=lambda s: s.quantile(0.05),
               sim_p95=lambda s: s.quantile(0.95), n_sim="count").reset_index())
spread["country"] = spread["variant"].map(lambda t: VARIANTS[t]["country"])

obs = pd.read_csv(OBSERVED_CSV)
obs = obs[(obs["version"] == OBSERVED_VERSION) & (obs["mass_col"] == SIM_MASS)]
obs = obs[["country", "model", "diff_H_median", "diff_H_lo", "diff_H_hi"]].rename(
    columns={"diff_H_median": "observed_median"})

summary = spread.merge(obs, on=["country", "model"], how="left")
summary["observed_inside_sim_range"] = (summary["observed_median"] >= summary["sim_p05"]) & \
                                       (summary["observed_median"] <= summary["sim_p95"])
summary = summary[["country", "model", "n_sim", "sim_median", "sim_p05", "sim_p95",
                   "observed_median", "observed_inside_sim_range"]]
summary.to_csv(SIM_SUMMARY_CSV, index=False)
print(summary.round(3).to_string(index=False))
print("failed fits in the simulation:", int((sim["error_class"].fillna("") != "").sum()))
print("Saved:", os.path.relpath(SIM_SUMMARY_CSV))